# 🔄 Принудительная конвертация MUSDB18

Этот ноутбук конвертирует `.stem.mp4` из MUSDB18 в WAV-формат
для обучения. Запускать, если основная конвертация не прошла.

**Почему отдельный ноутбук.** Скачанный MUSDB18 — это не папки с
дорожками, а плоский список `.stem.mp4`. Каждый такой файл не является
аудиозаписью: это контейнер сразу с 5 аудиопотоками. Основная ячейка
конвертации ищет папки и на таком датасете находит ноль треков, после
чего обучение молча уходит в синтетический режим (2 эпохи на шуме).

**Что получится на выходе:**

```
/content/custom_dataset_full/train/track_000/{mixture,vocals,drums,bass,other}.wav
/content/custom_dataset_full/valid/track_010/{mixture,vocals,drums,bass,other}.wav
```

После успешного запуска вернись в `colab_training_full.ipynb` и
запусти его заново начиная с ячейки 7 — он увидит готовые данные.

⚠️ **Важно про `test/`.** Если скачаны и `train/`, и `test/`, ноутбук
возьмёт только `train/`. Если файлы лежат плоско, а официальный
отладочный набор среди них — выведет предупреждение: часть треков
попадёт в обучение, и метрика на них будет нечестной.

In [ ]:
# Подключаем Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Клонируем код (если ещё не склонирован)
import os, shutil
os.chdir('/content')
PROJECT_DIR = '/content/demucs_custom'
if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/IgorCim/demucs_custom.git
os.chdir(PROJECT_DIR)
!git pull origin master
print(f"Работаем в: {os.getcwd()}")

In [ ]:
# Зависимости. ffmpeg обязателен: без него .stem.mp4 не распаковать —
# это контейнер с 5 потоками, а не аудиофайл.
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg
!pip install -q musdb
import shutil
assert shutil.which('ffmpeg'), "ffmpeg не установлен!"
print("✅ ffmpeg и библиотеки установлены")

In [ ]:
# Принудительная конвертация .stem.mp4 -> WAV
#
# Порядок потоков внутри .stem.mp4 задан в mus.yaml у demucs:
#     0 = mixture, 1 = drums, 2 = bass, 3 = other, 4 = vocals
# Названия потоков НЕ записаны внутрь файла, поэтому порядок задан
# жёстко и ошибиться в нём легко.
import os
import shutil
import subprocess
from pathlib import Path

# Очищаем папку назначения
target_dir = '/content/custom_dataset_full'
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)
Path(f'{target_dir}/train').mkdir(parents=True, exist_ok=True)
Path(f'{target_dir}/valid').mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Поиск исходников
# ---------------------------------------------------------------------------
source_dir = Path('/content/musdb18')

if not source_dir.is_dir():
    raise RuntimeError(
        f"{source_dir} не найден. Сначала скачай MUSDB18 в основном ноутбуке "
        "(musdb.DB(root='/content/musdb18', download=True))."
    )

all_mp4 = sorted(source_dir.rglob('*.stem.mp4'))


def in_official_test(path):
    """Лежит ли файл в официальной папке test/.

    Смотрим на КОМПОНЕНТЫ пути, а не на подстроку: трек с названием
    "Testify - Something.stem.mp4" не должен теряться из-за слова test.
    """
    rel = path.relative_to(source_dir)
    return 'test' in {part.lower() for part in rel.parts[:-1]}


mp4_files = [f for f in all_mp4 if not in_official_test(f)]
skipped_test = len(all_mp4) - len(mp4_files)

print(f"Всего .stem.mp4:        {len(all_mp4)}")
print(f"Исключено из test/:     {skipped_test}")
print(f"Найдено для конвертации: {len(mp4_files)}")

if not mp4_files:
    raise RuntimeError(
        "После исключения test/ не осталось ни одного файла — "
        "похоже, все скачанные треки лежат в test/. "
        "Проверь структуру папки и при необходимости убери фильтр."
    )

if skipped_test == 0 and len(all_mp4) > 100:
    # 144 = 100 train + 44 test у полного сэмпла MUSDB
    print()
    print("⚠️  ВНИМАНИЕ: папки test/ не найдено, файлы лежат плоско.")
    print("    В сэмпле MUSDB 144 файла = 100 train + 44 test.")
    print("    Похоже, в обучение попадут и официальные отладочные треки.")
    print("    Обучение будет работать, но метрика на них нечестная.")

# ---------------------------------------------------------------------------
# Распаковка
# ---------------------------------------------------------------------------
streams = {
    0: 'mixture.wav',
    1: 'drums.wav',
    2: 'bass.wav',
    3: 'other.wav',
    4: 'vocals.wav',
}

# Один общий gain на все пять файлов трека: пик суммы четырёх дорожек
# на настоящем материале выше 1.0, и при посегментной нормировке mixture
# разошлась бы с дорожками. Для .stem.mp4 не нужен — потоки уже в PCM_16.

failed = []
converted = 0

for i, mp4_path in enumerate(mp4_files):
    # 10% в valid. Именно по кругу, а не хвостом: имена треков идут
    # по алфавиту, и сплошной хвост собрал бы в valid одни "W-Z".
    split = 'valid' if i % 10 == 0 else 'train'
    track_dir = Path(target_dir) / split / f'track_{i:03d}'
    track_dir.mkdir(parents=True, exist_ok=True)

    for stream_idx, out_name in streams.items():
        out_path = track_dir / out_name
        cmd = [
            'ffmpeg', '-y', '-i', str(mp4_path),
            '-map', f'0:a:{stream_idx}',
            '-acodec', 'pcm_s16le',
            '-ar', '44100',
            '-ac', '2',
            str(out_path),
        ]
        result = subprocess.run(cmd, stdout=subprocess.DEVNULL,
                                stderr=subprocess.PIPE, text=True,
                                errors="replace")

        # Проверяем РЕЗУЛЬТАТ, а не только код возврата: если потока
        # с таким номером нет, ffmpeg молча ничего не создаёт, и трек
        # засчитался бы успешным, а данные потом оказались битыми.
        if result.returncode != 0 or not out_path.is_file() \
                or out_path.stat().st_size == 0:
            failed.append((mp4_path.name, out_name,
                           (result.stderr or '').strip()[-120:]))
            break

    if len(list(track_dir.glob('*.wav'))) == 5:
        converted += 1
    else:
        # Неполный трек удаляем: оставить его хуже, чем убрать.
        shutil.rmtree(track_dir, ignore_errors=True)

    if (i + 1) % 10 == 0:
        print(f"  обработано {i + 1}/{len(mp4_files)}")

if failed:
    print()
    print(f"⚠️  Не удалось распаковать: {len(failed)}")
    for name, out, err in failed[:5]:
        print(f"     {name} -> {out}: {err}")
    if len(failed) > 5:
        print(f"     ... и ещё {len(failed) - 5}")

# ---------------------------------------------------------------------------
# Итоги
# ---------------------------------------------------------------------------
train_count = len(list((Path(target_dir) / 'train').glob('track_*')))
valid_count = len(list((Path(target_dir) / 'valid').glob('track_*')))

print(f"\n✅ Конвертация завершена!")
print(f"   Треков в train: {train_count}")
print(f"   Треков в valid: {valid_count}")
print(f"   Из них с 5 файлами: {converted}")

# Проверка примера
train_dirs = sorted((Path(target_dir) / 'train').glob('track_*'))
if not train_dirs:
    raise RuntimeError("train пуст — обучение нормально не запустится.")

example = train_dirs[0]
files = sorted(f.name for f in example.iterdir())
print(f"   Пример структуры: {example.name}")
print(f"   Файлы: {files}")

# Финальная проверка: убеждаемся, что проект действительно это читает.
import sys
sys.path.insert(0, '/content/demucs_custom')
os.chdir('/content/demucs_custom')
import numpy as np
import soundfile as sf
from data_prep import CustomDataset

expected = sorted(['mixture.wav', 'vocals.wav', 'drums.wav', 'bass.wav', 'other.wav'])
assert files == expected, f"Не хватает файлов: {set(expected) - set(files)}"

# Сэмпл — треки по ~7 секунд. Сегмент 6 секунд: при 7.8 сек CustomDataset
# добивает трек нулями, и в каждом батче будет около секунды тишины.
ds = CustomDataset(root_dir=target_dir, split='train',
                   segment_length=264600, verbose=False)
item = ds[0]
info = sf.info(str(example / 'mixture.wav'))

print()
print("=" * 66)
print("ПРОВЕРКА ЧТЕНИЕМ")
print("=" * 66)
print(f"   Треков читается:     {len(ds)}")
print(f"   Формат mixture:      {tuple(item['mixture'].shape)}")
print(f"   Формат sources:      {tuple(item['sources'].shape)}")
print(f"   Длина трека:          {info.duration:.2f} сек")
print(f"   Частота/каналы:       {info.samplerate} Гц / {info.channels}")
print()
print(f"✅ Данные готовы. Вернись в colab_training_full.ipynb, ячейка 7.")
print("   segment_length для сэмпла = 264600 (6 сек), он уже выставлен там.")
print("=" * 66)